In [1]:
import os
from dotenv import load_dotenv
from FinMind.data import DataLoader
import pandas as pd
import json
import numpy as np
from hmmlearn.hmm import GaussianHMM
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

In [ ]:
def main():
    # read from argument parser (replace it with more common name if a more common name exist)
    # also add recommended default value for icovariance matrix and tol
    # Arguments: input file (string), outdir (string), iterations (int),
    # number of states (int), covariance matrix ("full"/ "diag"),
    # tol "tolerance for loglikelihood to stop fitting", random seed

    # check if the input file is ".npz"
    # create a folder for output file (exist_ok=True) under datapath, named output

    # Read data from the input file
    # store "lengths" (array of int), "concatenated observations" (array of array), "names" (array of strings)

    # create GaussianHMM model
    # use tqdm to show status bar and train the model

    # store the trained parameters and the names as npz and the argument as metadata.json
    # print the names and each of the emmission probabitily density distribution




## FINMIND API

In [43]:
stock_set = set()
for trader_id in [1360, 1400, 1440, 1470, 1480, 1560, 1650, 1660, 7030, 8440, 8960]:
    df = pd.read_parquet(f"../data/brokers/{trader_id}/2021-06-30_to_2026-02-11.parquet")
    # df[df["stock_id"] == "0050"]
    unique_values = set(df["stock_id"].unique())
    stock_set |= unique_values

In [48]:
with open("stock_ids.json", "w") as f:
    json.dump(list(stock_set), f, indent=4)

In [15]:
# df = pd.read_parquet("../data/stocks/0050_2021-06-30_to_2026-02-11.parquet")
# df = pd.read_parquet("../data/brokers/1440/2021-06-30_to_2026-02-11.parquet")
# df = pd.read_parquet("../data/preprocessed_data/final_vectors.parquet")
df = np.load("../data/preprocessed_data/hmm_lengths.npy")
df[:60]

array([ 35,   1,  66,  14,  33, 110,  26,  35,  12,  37,  74,  42,   4,
         5,   9,  52,  17,   9,  23,  74,  26,  55,  26,  90,   9,  10,
        11,  43,  10,  53,  18,  15,  15, 119,  74,  16, 120,  52,   8,
        11,   6,   1,  22,  31,  43,   9,  14,  70,  49,  12,   5,   9,
        52,  77,  16,   3,   3,   8,  34,   8])

In [ ]:
class BrokerTradeTracker:

    def __init__(self, securities_trader: str, securities_trader_id: str, stock_id: str) -> None:
        self.securities_trader: str = securities_trader
        self.securities_trader_id: str = securities_trader_id
        self.stock_id = stock_id

    def add_record(self, df: pd.DataFrame) -> None:
        self.date = df["date"]

In [3]:
import os
from dotenv import load_dotenv
import pandas as pd
from FinMind.data import DataLoader

# 載入 API key
load_dotenv()
token = os.environ["FINMIND_API_KEY"]

# 初始化 API
api = DataLoader()
api.login_by_token(api_token=token)

# 取得 1470 在 2026-03-02 的交易資料
df = api.taiwan_stock_trading_daily_report(
    securities_trader_id="1440",
    date="2026-03-02",
)

# 篩選華邦電 2344
df_2344 = df[df["stock_id"] == "2344"]

# 計算進出金額
if not df_2344.empty:
    df_2344["buy_amount"] = df_2344["buy"] * df_2344["price"]
    df_2344["sell_amount"] = df_2344["sell"] * df_2344["price"]
    df_2344["net_buy"] = df_2344["buy"] - df_2344["sell"]
    df_2344["net_buy_amount"] = df_2344["buy_amount"] - df_2344["sell_amount"]

print(df_2344)

2026-03-02 22:22:37.736 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-03-02 22:22:37.795 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-03-02 22:22:37.796 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockInfo, data_id: 
2026-03-02 22:22:38.201 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockPrice, data_id: 
2026-03-02 22:22:40.866 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTradingDailyReport, data_id: 


     securities_trader  price      buy    sell securities_trader_id stock_id  \
2348                美林  115.0     1000   80000                 1440     2344   
2349                美林  115.5        0  230000                 1440     2344   
2350                美林  116.0        0   51000                 1440     2344   
2351                美林  117.0        0   89000                 1440     2344   
2352                美林  117.5        0  275600                 1440     2344   
2353                美林  118.0        0   97000                 1440     2344   
2354                美林  119.5        0  137000                 1440     2344   
2355                美林  120.0   152000    1000                 1440     2344   
2356                美林  120.5   619000       0                 1440     2344   
2357                美林  121.0    93000       0                 1440     2344   
2358                美林  121.5    67000   74000                 1440     2344   
2359                美林  122.0    38000  

In [6]:
df_3037 = df[df["stock_id"] == "3037"]


In [ ]:
df_3037.to_csv("test.csv", index=False, encoding="utf-8-sig")

: 

In [9]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)
print(df_3037)

     securities_trader  price    buy   sell securities_trader_id stock_id        date
5130                美林  463.5  77000      0                 1440     3037  2026-03-02
5131                美林  467.0   1000      0                 1440     3037  2026-03-02
5132                美林  467.5   3000      0                 1440     3037  2026-03-02
5133                美林  469.0   4000      0                 1440     3037  2026-03-02
5134                美林  469.5   1000      0                 1440     3037  2026-03-02
...                ...    ...    ...    ...                  ...      ...         ...
5191                美林  499.5  22000      0                 1440     3037  2026-03-02
5192                美林  500.0  14000  54000                 1440     3037  2026-03-02
5193                美林  501.0  20000   1000                 1440     3037  2026-03-02
5194                美林  502.0   3000      0                 1440     3037  2026-03-02
5195                美林  503.0   8000      0           